# 1. Khởi tạo mô hình và điều chỉnh lớp phân loại

In [ ]:
from torch.nn.modules.activation import Hardswish
from torchvision import models
import torch.nn as nn

model = models.mobilenet_v3_small(weights= "IMAGENET1K_V1")

model.classifier = nn.Sequential(
    nn.Linear(576, 1024),
    nn.Hardswish(),
    nn.Dropout(p=0.2),
    nn.Linear(1024, 256),
)

print(model.classifier)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 150MB/s]

Sequential(
  (0): Linear(in_features=576, out_features=1024, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=1024, out_features=256, bias=True)
)


# 2. Tiền xử lý theo chuẩn PyTorch

In [ ]:
from torchvision import transforms, datasets

# Chuẩn hóa cho tập train
transform_train = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

])

# Chuẩn hóa cho tập val và test
transform_val = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Cấu hình training (TripletDataset)

In [ ]:
# TripletDataset

from torch.utils.data import Dataset, DataLoader
import random

class TripletDataset(Dataset):
  def __init__(self, dataset):
    self.dataset = dataset
    self.targets = dataset.targets
    self.class_to_indices = {}
    for idx, target in enumerate(self.targets):
      self.class_to_indices.setdefault(target, []).append(idx)

  def __getitem__(self, index):
    anchor_img, anchor_lable = self.dataset[index]
    positive_index = random.choice(self.class_to_indices[anchor_lable])
    negative_label = random.choice([lbl for lbl in self.class_to_indices.keys() if lbl != anchor_lable])
    negative_index = random.choice(self.class_to_indices[negative_label])

    pos_img, _ = self.dataset[positive_index]
    neg_img, _ = self.dataset[negative_index]
    return anchor_img, pos_img, neg_img


  def __len__(self):
    return len(self.dataset)

In [ ]:
# TripletLoss
import torch
import torch.optim as optim

criterion = nn.TripletMarginLoss(margin=1.0, p=2)

# Chỉ fine tune lớp head
for param in model.features.parameters():
  param.requires_grad = False
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(device)

cuda


# 4. Phân chia tập dữ liệu

In [ ]:
# !pip install split-folders

In [ ]:
# import splitfolders

# input_folder = "/content/drive/MyDrive/DATA_NEW"

# output_folder = "/content/drive/MyDrive/DATASET"

# splitfolders.ratio(
#     input=input_folder,
#     output=output_folder,
#     seed=42,
#     ratio=(0.8, 0.1, 0.1),
#     group_prefix=None
# )

# 5. Khởi tạo dataloader

In [ ]:
train_dir = "/content/drive/MyDrive/DATASET/train"
val_dir = "/content/drive/MyDrive/DATASET/val"

# Tải dữ liệu thô
train_dataset = datasets.ImageFolder(train_dir, transform=transform_train)
val_dataset = datasets.ImageFolder(val_dir, transform=transform_val)

# Tạo tripletdataset
train_triplet_ds = TripletDataset(train_dataset)
val_triplet_ds = TripletDataset(val_dataset)

# Khởi tạo dataloader
train_loader = DataLoader(
    dataset=train_triplet_ds,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)
val_loader = DataLoader(
    dataset=val_triplet_ds,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)



# 6. Xây dựng vòng lặp huấn luyện và đánh giá

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

def train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=15, save_path="best_model.pth"):
    print(f"🚀 Starting training on {device}")
    best_val_loss = float("inf")

    for epoch in range(num_epochs):
        ########################
        # 1. Training
        ########################
        model.train()
        train_loss = 0.0
        for i, (anchor, positive, negative) in enumerate(train_loader):
            anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)

            emb_anchor = model(anchor)
            emb_pos = model(positive)
            emb_neg = model(negative)

            loss = criterion(emb_anchor, emb_pos, emb_neg)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            if (i+1) % 30 == 0:
                print(f"Epoch {epoch+1}/{num_epochs} | Batch {i+1}/{len(train_loader)} - Loss: {loss.item():.4f}")

        avg_train_loss = train_loss / len(train_loader)

        ########################
        # 2. Validation
        ########################
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for anchor, positive, negative in val_loader:
                anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)

                emb_anchor = model(anchor)
                emb_pos = model(positive)
                emb_neg = model(negative)

                loss = criterion(emb_anchor, emb_pos, emb_neg)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)

        ########################
        # 3. Ghi lại tiến trình
        ########################
        print(f"- Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}\n")

        ########################
        # 4. Lưu model tốt nhất
        ########################
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), save_path)
            print(f"💾 Saved best model at epoch {epoch+1} with Val Loss: {avg_val_loss:.4f}")

    print("✅ Training finished!")
    print(f"Best validation loss: {best_val_loss:.4f}")


In [ ]:
# Thực hiện huấn luyện
train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=15 )

🚀 Starting training on cuda


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1/15 | Batch 30/99 - Loss: 0.6249
Epoch 1/15 | Batch 60/99 - Loss: 0.4693
Epoch 1/15 | Batch 90/99 - Loss: 0.3658
- Epoch [1/15] Train Loss: 0.4793 | Val Loss: 0.4264

💾 Saved best model at epoch 1 with Val Loss: 0.4264
Epoch 2/15 | Batch 30/99 - Loss: 0.4781
Epoch 2/15 | Batch 60/99 - Loss: 0.4838
Epoch 2/15 | Batch 90/99 - Loss: 0.2447
- Epoch [2/15] Train Loss: 0.4475 | Val Loss: 0.3870

💾 Saved best model at epoch 2 with Val Loss: 0.3870
Epoch 3/15 | Batch 30/99 - Loss: 0.5234
Epoch 3/15 | Batch 60/99 - Loss: 0.3724
Epoch 3/15 | Batch 90/99 - Loss: 0.5286
- Epoch [3/15] Train Loss: 0.4269 | Val Loss: 0.3451

💾 Saved best model at epoch 3 with Val Loss: 0.3451
Epoch 4/15 | Batch 30/99 - Loss: 0.6094
Epoch 4/15 | Batch 60/99 - Loss: 0.3401
Epoch 4/15 | Batch 90/99 - Loss: 0.2079
- Epoch [4/15] Train Loss: 0.4045 | Val Loss: 0.3780

Epoch 5/15 | Batch 30/99 - Loss: 0.3580
Epoch 5/15 | Batch 60/99 - Loss: 0.3114
Epoch 5/15 | Batch 90/99 - Loss: 0.5412
- Epoch [5/15] Train Loss: 0